<a href="https://colab.research.google.com/github/aniray2908/nlp-llm-journey/blob/main/01_nlp_fundamentals/demos/02_text_normalisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1 — Imports & Setup**

In [1]:
!pip install nltk -q

import nltk
import re
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pandas as pd

# Download required data
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

**Cell 2 — Porter Stemmer Basics**

In [2]:
stemmer = PorterStemmer()

words = [
    "running", "runs", "ran", "runner",
    "love", "loves", "loving", "loved",
    "better", "good",
    "caresses", "ponies", "sing", "singular",
    "relational", "relate",
]

results = []
for word in words:
    stem = stemmer.stem(word)
    results.append({"word": word, "stem": stem})

df = pd.DataFrame(results)
print(df.to_string(index=False))

      word     stem
   running      run
      runs      run
       ran      ran
    runner   runner
      love     love
     loves     love
    loving     love
     loved     love
    better   better
      good     good
  caresses   caress
    ponies     poni
      sing     sing
  singular singular
relational    relat
    relate    relat


**Cell 3 — WordNet Lemmatiser Basics**

In [3]:
lemmatizer = WordNetLemmatizer()

words = [
    "running", "runs", "ran", "runner",
    "love", "loves", "loving", "loved",
    "better", "good",
    "were", "is", "are",
]

results = []
for word in words:
    # Try as verb first (most common case)
    lemma_v = lemmatizer.lemmatize(word, pos='v')
    # Also try as noun
    lemma_n = lemmatizer.lemmatize(word, pos='n')
    results.append({
        "word": word,
        "lemma (verb)": lemma_v,
        "lemma (noun)": lemma_n
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

   word lemma (verb) lemma (noun)
running          run      running
   runs          run          run
    ran          run          ran
 runner       runner       runner
   love         love         love
  loves         love         love
 loving         love       loving
  loved         love        loved
 better       better       better
   good         good         good
   were           be         were
     is           be           is
    are           be          are


**Cell 4 — Stemming vs Lemmatisation Comparison**

In [4]:
test_words = [
    "running", "better", "were", "caresses", "ponies",
    "runner", "loving", "singular", "relational"
]

comparison = []
for word in test_words:
    stem = stemmer.stem(word)
    lemma = lemmatizer.lemmatize(word, pos='v')
    comparison.append({
        "word": word,
        "stem": stem,
        "lemma": lemma,
        "stem==lemma": stem == lemma
    })

df = pd.DataFrame(comparison)
print(df.to_string(index=False))

print(f"\nAverage aggressiveness:")
print(f"  Stemming removes {sum(len(w) - len(stemmer.stem(w)) for w in test_words) / len(test_words):.1f} chars/word")
print(f"  Lemmatising removes {sum(len(w) - len(lemmatizer.lemmatize(w, pos='v')) for w in test_words) / len(test_words):.1f} chars/word")

      word     stem      lemma  stem==lemma
   running      run        run         True
    better   better     better         True
      were     were         be        False
  caresses   caress     caress         True
    ponies     poni     ponies        False
    runner   runner     runner         True
    loving     love       love         True
  singular singular   singular         True
relational    relat relational        False

Average aggressiveness:
  Stemming removes 1.7 chars/word
  Lemmatising removes 1.1 chars/word


**Cell 5 — Stopword Removal**

In [5]:
stop_words = set(stopwords.words('english'))

print(f"Total English stopwords: {len(stop_words)}")
print(f"First 20: {sorted(stop_words)[:20]}")

# Apply to a sentence
sentence = "The quick brown fox jumps over the lazy dog"
tokens = sentence.lower().split()

filtered = [w for w in tokens if w not in stop_words]

print(f"\nOriginal: {sentence}")
print(f"Tokens: {tokens}")
print(f"After stopword removal: {filtered}")
print(f"Removed: {set(tokens) - set(filtered)}")

Total English stopwords: 198
First 20: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']

Original: The quick brown fox jumps over the lazy dog
Tokens: ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
After stopword removal: ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog']
Removed: {'over', 'the'}


**Cell 6 — The Stopword Problem**

In [6]:
sentences = [
    "I love this movie",
    "I do not love this movie",
    "This is good",
    "This is not good",
    "The cat sat on the mat",
    "The cat did not sit on the mat",
]

stop_words = set(stopwords.words('english'))

print(f"{'Original':<35} {'After stopword removal':<35} {'OK?'}")
print("-" * 75)
for sent in sentences:
    tokens = sent.lower().split()
    filtered = ' '.join([w for w in tokens if w not in stop_words])
    ok = "✓" if "not" not in filtered else "✗"
    print(f"{sent:<35} {filtered:<35} {ok}")

Original                            After stopword removal              OK?
---------------------------------------------------------------------------
I love this movie                   love movie                          ✓
I do not love this movie            love movie                          ✓
This is good                        good                                ✓
This is not good                    good                                ✓
The cat sat on the mat              cat sat mat                         ✓
The cat did not sit on the mat      cat sit mat                         ✓


**Cell 7 — Complete Text Normalisation Pipeline**

In [8]:
nltk.download('punkt_tab')
def normalise_text(text, lowercase=True, remove_special=True,
                   remove_stopwords=False, lemmatise=False):
    """
    Complete text normalisation pipeline.

    Args:
        text: input text
        lowercase: convert to lowercase
        remove_special: remove punctuation and numbers
        remove_stopwords: filter stopwords
        lemmatise: lemmatise words

    Returns:
        list of normalised tokens
    """
    # 1. Lowercase
    if lowercase:
        text = text.lower()

    # 2. Remove special characters and numbers
    if remove_special:
        text = re.sub(r'[^a-z\s]', '', text) if lowercase else re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Tokenise
    tokens = word_tokenize(text)

    # 4. Remove stopwords
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [t for t in tokens if t not in stop_words]

    # 5. Lemmatise
    if lemmatise:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(t, pos='v') for t in tokens]

    return tokens


# Test on various texts
test_texts = [
    "I LOVE ChatGPT!!! It's AMAZING... Running tests now!",
    "The quick BROWN fox jumps over the lazy DOG.",
    "Don't worry about spacing...   it'll be fixed!!!",
    "I'm loving it, and you should too!",
]

configs = [
    {"name": "Raw tokenisation", "lowercase": True, "remove_special": False, "remove_stopwords": False, "lemmatise": False},
    {"name": "Clean + lowercase", "lowercase": True, "remove_special": True, "remove_stopwords": False, "lemmatise": False},
    {"name": "No stopwords", "lowercase": True, "remove_special": True, "remove_stopwords": True, "lemmatise": False},
    {"name": "With lemmatisation", "lowercase": True, "remove_special": True, "remove_stopwords": False, "lemmatise": True},
    {"name": "Full pipeline", "lowercase": True, "remove_special": True, "remove_stopwords": True, "lemmatise": True},
]

for text in test_texts:
    print(f"\nOriginal: {text}")
    print("-" * 80)
    for config in configs:
        result = normalise_text(text, **{k: v for k, v in config.items() if k != 'name'})
        print(f"  {config['name']:<20} → {result}")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.



Original: I LOVE ChatGPT!!! It's AMAZING... Running tests now!
--------------------------------------------------------------------------------
  Raw tokenisation     → ['i', 'love', 'chatgpt', '!', '!', '!', 'it', "'s", 'amazing', '...', 'running', 'tests', 'now', '!']
  Clean + lowercase    → ['i', 'love', 'chatgpt', 'its', 'amazing', 'running', 'tests', 'now']
  No stopwords         → ['love', 'chatgpt', 'amazing', 'running', 'tests']
  With lemmatisation   → ['i', 'love', 'chatgpt', 'its', 'amaze', 'run', 'test', 'now']
  Full pipeline        → ['love', 'chatgpt', 'amaze', 'run', 'test']

Original: The quick BROWN fox jumps over the lazy DOG.
--------------------------------------------------------------------------------
  Raw tokenisation     → ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']
  Clean + lowercase    → ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
  No stopwords         → ['quick', 'brown', 'fox', 'jumps', 'laz

**Cell 8 — Real-World Impact: Sentiment Analysis**

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Fake sentiment data
texts = [
    "I love this movie, it's amazing!",
    "This movie is terrible, I hated it.",
    "Running is great, I love running!",
    "Not great, not terrible, just okay.",
    "The best film I've ever seen!",
    "The worst waste of time ever.",
    "I'm running away from this bad movie!",
    "Love the characters and the plot!",
]

labels = [1, 0, 1, 0, 1, 0, 0, 1]  # 1=positive, 0=negative

# Pipeline 1: Raw text
vectorizer_raw = TfidfVectorizer(lowercase=True, token_pattern=r'\b\w+\b')
X_raw = vectorizer_raw.fit_transform(texts)

# Pipeline 2: With stopword removal
vectorizer_stop = TfidfVectorizer(lowercase=True, stop_words='english', token_pattern=r'\b\w+\b')
X_stop = vectorizer_stop.fit_transform(texts)

# Train simple classifier on each
clf_raw = MultinomialNB().fit(X_raw, labels)
clf_stop = MultinomialNB().fit(X_stop, labels)

acc_raw = clf_raw.score(X_raw, labels)
acc_stop = clf_stop.score(X_stop, labels)

print(f"Accuracy with raw text: {acc_raw:.1%}")
print(f"Accuracy with stopword removal: {acc_stop:.1%}")

# Show what features are most important
print(f"\nTop features (raw): {vectorizer_raw.get_feature_names_out()[:10]}")
print(f"Top features (stopwords removed): {vectorizer_stop.get_feature_names_out()[:10]}")

Accuracy with raw text: 100.0%
Accuracy with stopword removal: 100.0%

Top features (raw): ['amazing' 'and' 'away' 'bad' 'best' 'characters' 'ever' 'film' 'from'
 'great']
Top features (stopwords removed): ['amazing' 'away' 'bad' 'best' 'characters' 'film' 'great' 'hated' 'just'
 'love']


**Cell 9 — Stemming Failures Gallery**

In [10]:
failure_cases = [
    ("ran", "verb"),
    ("singular", "adjective"),
    ("relational", "adjective"),
    ("caresses", "noun"),
    ("dying", "verb"),
    ("lying", "verb"),
    ("tying", "verb"),
    ("conflated", "verb"),
    ("troubled", "verb"),
]

print(f"{'Word':<15} {'POS':<12} {'Stem':<15} {'Issue'}")
print("-" * 60)
for word, pos in failure_cases:
    stem = stemmer.stem(word)
    # Try to categorise the failure
    if stem == word:
        issue = "No stem (too short?)"
    elif not stem.endswith(word.rstrip('edsing')):
        issue = "Over-stemmed"
    else:
        issue = "Irregular/edge case"

    print(f"{word:<15} {pos:<12} {stem:<15} {issue}")

Word            POS          Stem            Issue
------------------------------------------------------------
ran             verb         ran             No stem (too short?)
singular        adjective    singular        No stem (too short?)
relational      adjective    relat           Over-stemmed
caresses        noun         caress          Over-stemmed
dying           verb         die             Over-stemmed
lying           verb         lie             Over-stemmed
tying           verb         tie             Over-stemmed
conflated       verb         conflat         Irregular/edge case
troubled        verb         troubl          Irregular/edge case


**Cell 10 — When Lemmatisation Matters**

In [11]:
# Example where lemmatisation is valuable
text = """
The runners are running in the race. They ran fast yesterday.
The running shoes help them run better. The best runner won.
"""

tokens = word_tokenize(text.lower())

# Without lemmatisation
from collections import Counter
freq_raw = Counter(tokens)
run_variants_raw = {w: c for w, c in freq_raw.items() if 'run' in w}

# With lemmatisation
lemmatizer = WordNetLemmatizer()
lemmas = [lemmatizer.lemmatize(t, pos='v') for t in tokens]
freq_lemma = Counter(lemmas)
run_variants_lemma = {w: c for w, c in freq_lemma.items() if 'run' in w}

print("Without lemmatisation:")
print(f"  Variants: {run_variants_raw}")
print(f"  Total 'run' family: {sum(run_variants_raw.values())}")

print("\nWith lemmatisation:")
print(f"  Variants: {run_variants_lemma}")
print(f"  Total 'run' family: {sum(run_variants_lemma.values())}")

Without lemmatisation:
  Variants: {'runners': 1, 'running': 2, 'run': 1, 'runner': 1}
  Total 'run' family: 5

With lemmatisation:
  Variants: {'runners': 1, 'run': 4, 'runner': 1}
  Total 'run' family: 6
